# Egyptian Auction Price Prediction — End-to-End ML Project

**Problem:** Regression — predict `final_selling_price` (EGP)  
**Dataset:** 20,000 rows × 18 columns  
**Best Model:** Ensemble (RF + LightGBM + XGBoost), log-target  
**Final Test Performance:** R² = 0.9372 | RMSLE = 0.2674 | Median % Error = 19.5%

> **Why RMSLE?** Prices span 50–1.4M EGP. RMSLE penalises relative errors equally  
> across all scales — superior to MAE/RMSE for wide-range price prediction.


## Setup

In [ ]:
import warnings, sys, os
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
%matplotlib inline
print("Libraries loaded.")

---
## Phase 1 — Exhaustive EDA
Full coverage: shape, dtypes, missing values, duplicates, cardinality, outliers (IQR + Z-score),
distribution (skew/kurtosis), **data leakage detection**, multicollinearity (VIF + heatmap),
bivariate analysis (scatter, violin, chi-square, Cramers V), formatting issues, impossible values.


In [ ]:
from scripts.data_exploration import run_eda
df, dist_stats, outlier_report = run_eda()
print(f"EDA complete: {df.shape[0]:,} rows x {df.shape[1]} columns")

### Key EDA Findings

| Finding | Detection | Fix in Phase 2 |
|---------|-----------|----------------|
| `reserve_price` corr=0.990 with target | Pearson r | DROP — leakage |
| `buy_now_price` corr=0.993 with target | Pearson r | DROP — leakage |
| `item_title`, `item_description` near-unique | nunique=128 | DROP — free text |
| `starting_price` skew=+5.44 | .skew() | log1p transform |
| `seller_total_sales` skew=+7.67 | .skew() | log1p transform |
| `seller_account_age` skew=+1.21 | .skew() | log1p transform |
| `starting_price` 2283 outlier rows | IQR | Clip 1st-99th pct |
| `brand` cardinality=126 | nunique | Frequency encoding |
| No missing values | isnull().sum() | No imputation needed |
| No duplicates | .duplicated() | No action |
| No impossible values | Range checks | No action |


---
## Phase 2 — Data Preprocessing

10 ordered steps (order is non-negotiable — every position is justified):

| Step | Action | Why here? |
|------|--------|-----------|
| 1 | Drop duplicates | Before any transform |
| 2 | Drop leakage cols | Early — no wasted compute |
| 3 | Drop text/ID cols | Before encoding |
| 4 | Fix formatting | Before encoding — consistent maps |
| 5 | Handle missing | After formatting — clean values |
| 6 | Encode categoricals | Before outlier cap |
| 7 | Cap outliers (1st-99th pct) | Before log — prevent blowup |
| 8 | Log-transform skewed features | Before split — uniform |
| 9 | **Train/test split** | **Before scaling — prevents leakage** |
| 10 | **Scale (fit train only)** | **Transform-only on test** |


In [ ]:
from scripts.preprocessing import (
    load_raw, step1_drop_duplicates, step2_drop_leakage,
    step3_drop_text_cols, step4_fix_formatting, step5_handle_missing,
    step6_encode, step7_handle_outliers, step8_log_transform)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from scripts.utils import save_object

df_raw = load_raw()
df = step1_drop_duplicates(df_raw.copy())
df = step2_drop_leakage(df)
df = step3_drop_text_cols(df)
df = step4_fix_formatting(df)
df = step5_handle_missing(df)
df, encoders = step6_encode(df, fit=True)
df, caps     = step7_handle_outliers(df, fit=True)
df, log_cols = step8_log_transform(df)
print(f"Post-preprocessing shape: {df.shape}")

---
## Phase 4 — Feature Engineering

9 new features from listing-time columns (zero leakage):

In [ ]:
from scripts.feature_engineering import engineer_features
df = engineer_features(df)
print(f"Post-FE shape: {df.shape}")
print("Features:", df.columns.tolist())

### Split → Scale

In [ ]:
TARGET = "final_selling_price"
X_all = df.drop(columns=[TARGET])
y_all = df[TARGET]

# SPLIT BEFORE SCALING (mandatory — prevents test leakage into scaler)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42)

# RobustScaler: uses median+IQR, robust to residual outliers
# fit_transform on TRAIN, transform-only on TEST
scaler = RobustScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train_raw),
                          columns=X_train_raw.columns, index=X_train_raw.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test_raw),
                          columns=X_test_raw.columns, index=X_test_raw.index)

save_object(encoders, "encoders.pkl")
save_object(caps,     "outlier_caps.pkl")
save_object(log_cols, "log_cols.pkl")
save_object(scaler,   "scaler.pkl")
print(f"Train: {X_train_s.shape}  |  Test: {X_test_s.shape}")
print("Scaler fitted on train ONLY. Test: transform-only.")

---
## Phase 3 — Feature Selection
Applied on X_train only. Same mask applied to X_test (no refit).

In [ ]:
from scripts.feature_selection import select_features

X_train_sel, X_test_sel, sel_feats, report = select_features(
    X_train_s, X_test_s, y_train)

dropped = [c for c in X_train_s.columns if c not in sel_feats]
print(f"KEPT ({len(sel_feats)}):    {sel_feats}")
print(f"DROPPED ({len(dropped)}): {dropped}")

---
## Phase 5 — Modelling

### Why These 3 Models?
- **Random Forest:** robust ensemble baseline, handles non-linearities, low overfit risk
- **LightGBM:** fast, leaf-wise growth captures complex patterns, native regularisation
- **XGBoost:** state-of-the-art boosting, excellent regularisation, best for log-target

### Log-Target Strategy
Training on `log1p(price)` converts RMSLE minimisation into MSE in log-space.
After prediction: `expm1()` recovers original scale.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from scripts.utils import report_metrics, save_object
import numpy as np

log_y_train = np.log1p(y_train)

RF_PARAMS   = dict(n_estimators=300, max_depth=None, min_samples_leaf=1,
                   max_features=0.6, n_jobs=-1, random_state=42)
LGBM_PARAMS = dict(n_estimators=800, num_leaves=63, learning_rate=0.025,
                   subsample=0.85, colsample_bytree=0.7, reg_alpha=0.01,
                   reg_lambda=0.1, min_child_samples=10,
                   random_state=42, n_jobs=-1, verbose=-1)
XGB_PARAMS  = dict(n_estimators=600, max_depth=6, learning_rate=0.025,
                   subsample=0.85, colsample_bytree=0.7, reg_alpha=0.01,
                   reg_lambda=1.0, min_child_weight=5,
                   random_state=42, n_jobs=-1, verbosity=0)

results = {}

print("BASELINE (raw target)")
rf_b = RandomForestRegressor(**RF_PARAMS); rf_b.fit(X_train_sel, y_train)
results["RF_Baseline_Train"]   = report_metrics(y_train, rf_b.predict(X_train_sel), "RF_Base_Tr")
results["RF_Baseline_Test"]    = report_metrics(y_test,  rf_b.predict(X_test_sel),  "RF_Base_Te")

lgbm_b = LGBMRegressor(**LGBM_PARAMS); lgbm_b.fit(X_train_sel, y_train)
results["LGBM_Baseline_Train"] = report_metrics(y_train, lgbm_b.predict(X_train_sel), "LGBM_Base_Tr")
results["LGBM_Baseline_Test"]  = report_metrics(y_test,  lgbm_b.predict(X_test_sel),  "LGBM_Base_Te")

xgb_b = XGBRegressor(**XGB_PARAMS); xgb_b.fit(X_train_sel, y_train, verbose=False)
results["XGB_Baseline_Train"]  = report_metrics(y_train, xgb_b.predict(X_train_sel), "XGB_Base_Tr")
results["XGB_Baseline_Test"]   = report_metrics(y_test,  xgb_b.predict(X_test_sel),  "XGB_Base_Te")

In [ ]:
print("TUNED (log-target)")
rf_t = RandomForestRegressor(**RF_PARAMS); rf_t.fit(X_train_sel, log_y_train)
p_rf_tr = np.expm1(rf_t.predict(X_train_sel)); p_rf_te = np.expm1(rf_t.predict(X_test_sel))
results["RF_Tuned_Train"] = report_metrics(y_train, p_rf_tr, "RF_Tuned_Tr")
results["RF_Tuned_Test"]  = report_metrics(y_test,  p_rf_te, "RF_Tuned_Te")

lgbm_t = LGBMRegressor(**LGBM_PARAMS); lgbm_t.fit(X_train_sel, log_y_train)
p_lgbm_tr = np.expm1(lgbm_t.predict(X_train_sel)); p_lgbm_te = np.expm1(lgbm_t.predict(X_test_sel))
results["LGBM_Tuned_Train"] = report_metrics(y_train, p_lgbm_tr, "LGBM_Tuned_Tr")
results["LGBM_Tuned_Test"]  = report_metrics(y_test,  p_lgbm_te, "LGBM_Tuned_Te")

xgb_t = XGBRegressor(**XGB_PARAMS); xgb_t.fit(X_train_sel, log_y_train, verbose=False)
p_xgb_tr = np.expm1(xgb_t.predict(X_train_sel)); p_xgb_te = np.expm1(xgb_t.predict(X_test_sel))
results["XGB_Tuned_Train"] = report_metrics(y_train, p_xgb_tr, "XGB_Tuned_Tr")
results["XGB_Tuned_Test"]  = report_metrics(y_test,  p_xgb_te, "XGB_Tuned_Te")

In [ ]:
print("ENSEMBLE (RF+LGBM+XGB log-blend)")
def ens_predict(X):
    p = (rf_t.predict(X) + lgbm_t.predict(X) + xgb_t.predict(X)) / 3
    return np.maximum(np.expm1(p), 0)

p_ens_tr = ens_predict(X_train_sel)
p_ens_te = ens_predict(X_test_sel)
results["Ensemble_Train"] = report_metrics(y_train, p_ens_tr, "Ensemble_Tr")
results["Ensemble_Test"]  = report_metrics(y_test,  p_ens_te, "Ensemble_Te")

bundle = {"rf": rf_t, "lgbm": lgbm_t, "xgb": xgb_t, "log_target": True, "type": "ensemble_log_blend"}
save_object(bundle, "best_model.pkl")
save_object("Ensemble_LogBlend", "best_model_name.pkl")
print("Best model bundle saved.")

In [ ]:
# COMPARISON TABLE
rows = [{"Run": k, "R2": round(v["R2"],4), "RMSLE": round(v["RMSLE"],4)}
        for k,v in results.items()]
res_df = pd.DataFrame(rows)
print("=" * 55)
print(" FINAL MODEL COMPARISON TABLE")
print("=" * 55)
print(res_df.to_string(index=False))
res_df.to_csv("../outputs/model_results.csv", index=False)

best = res_df[res_df["Run"].str.contains("Test")].loc[
       res_df[res_df["Run"].str.contains("Test")]["RMSLE"].idxmin()]
print(f"\nBEST: {best['Run']}  R2={best['R2']}  RMSLE={best['RMSLE']}")

### Overfitting Analysis

| Model | Train R² | Test R² | Gap | Status |
|-------|----------|---------|-----|--------|
| RF Baseline | 0.9913 | 0.9334 | 0.058 | Acceptable |
| LGBM Baseline | 0.9977 | 0.9234 | 0.074 | Overfit on raw |
| XGB Baseline | 0.9924 | 0.9259 | 0.066 | Overfit on raw |
| RF Tuned | 0.9905 | 0.9351 | 0.055 | Good |
| LGBM Tuned | 0.9705 | 0.9356 | 0.035 | Excellent |
| XGB Tuned | 0.9647 | 0.9352 | 0.030 | Excellent |
| **Ensemble** | **0.9773** | **0.9372** | **0.040** | **Best** |


---
## Phase 6 — Final Predictions

Full fitted pipeline applied to test set — all transformers are transform-only (no refit).

In [ ]:
from scripts.predict import run_predictions
y_test_out, y_pred_out, final_metrics = run_predictions()
print(f"Final Test  R2={final_metrics['R2']:.4f}  RMSLE={final_metrics['RMSLE']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_out, y_pred_out, alpha=0.15, s=6, color="steelblue")
mn, mx = float(y_test_out.min()), float(y_test_out.max())
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect fit")
axes[0].set_xlabel("Actual Price (EGP)"); axes[0].set_ylabel("Predicted Price (EGP)")
axes[0].set_title(f"Actual vs Predicted\nR2={final_metrics['R2']:.4f}  RMSLE={final_metrics['RMSLE']:.4f}")
axes[0].legend()

axes[1].scatter(y_pred_out, y_test_out.values - y_pred_out, alpha=0.15, s=6, color="salmon")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Predicted")
plt.tight_layout()
plt.savefig("../outputs/final_predictions_notebook.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
comp = pd.read_csv("../outputs/predictions_comparison.csv")
print("True vs Predicted (30 test samples):")
print(comp.head(30).to_string(index=False))

---
## Summary

| Phase | Result |
|-------|--------|
| EDA | No missing/duplicate data; 2 leakage cols detected |
| Preprocessing | 10-step pipeline; all objects saved as .pkl |
| Feature Engineering | 10 new features |
| Feature Selection | 15 of 23 kept |
| Best Model | Ensemble R²=0.9372, RMSLE=0.2674 |
| API | FastAPI /predict endpoint |

**Pipeline guarantees:** No scaling before split ✓ | No fit on test ✓ | No leakage features ✓
